# Phase 3 – Sentiment Analysis with LinearSVC and Word2Vec

This notebook enhances the sentiment analysis from Phase 2 by training a **LinearSVC classifier**
using **Word2Vec embeddings** as features. We compare this supervised approach against the
existing lexicon-based method and analyze feature importance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
print("All imports loaded successfully.")

## 1. Load Data and Existing Sentiment Results

In [ ]:
# Load processed reviews and lexicon-based sentiment results
sentiment_df = pd.read_csv("data/sentiment_results.csv")

print(f"Total reviews: {len(sentiment_df)}")
print(f"\nColumns: {list(sentiment_df.columns)}")
print(f"\nLexicon sentiment distribution:")
print(sentiment_df['lexicon_sentiment'].value_counts())
print(f"\nSample reviews:")
sentiment_df.head()

## 2. Train Word2Vec Model and Generate Review Embeddings

In [ ]:
# Tokenize all processed reviews for Word2Vec
tokenized_reviews = [word_tokenize(str(review)) for review in sentiment_df['Processed_Review']]

# Train Word2Vec model (CBOW) matching Phase 2 parameters
w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    workers=4,
    epochs=50
)

print(f"Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimensions: {w2v_model.vector_size}")

In [ ]:
def get_review_vector(tokens, model):
    """Compute review embedding by averaging word vectors."""
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Generate embeddings for all reviews
review_vectors = np.array([
    get_review_vector(tokens, w2v_model)
    for tokens in tokenized_reviews
])

print(f"Review embeddings shape: {review_vectors.shape}")
print(f"Reviews with zero vectors: {np.sum(np.all(review_vectors == 0, axis=1))}")

## 3. Prepare Labels and Train LinearSVC Classifier

In [ ]:
# Encode sentiment labels
le = LabelEncoder()
labels = le.fit_transform(sentiment_df['lexicon_sentiment'])
class_names = le.classes_

print(f"Classes: {class_names}")
print(f"Label distribution: {dict(zip(class_names, np.bincount(labels)))}")

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    review_vectors, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"\nTraining set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Train label distribution: {dict(zip(class_names, np.bincount(y_train)))}")
print(f"Test label distribution: {dict(zip(class_names, np.bincount(y_test)))}")

In [ ]:
# Train LinearSVC with class weight balancing for imbalanced data
svc_model = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=10000,
    random_state=42
)
svc_model.fit(X_train, y_train)

# Wrap with CalibratedClassifierCV for probability/confidence estimates
calibrated_svc = CalibratedClassifierCV(svc_model, cv=5)
calibrated_svc.fit(X_train, y_train)

print("LinearSVC model trained successfully.")
print(f"Number of features: {svc_model.coef_.shape[1]}")
print(f"Number of classes: {len(class_names)}")

## 4. Model Evaluation

In [ ]:
# Predictions on test set
y_pred = svc_model.predict(X_test)

# Classification metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print("=" * 50)
print("LinearSVC Model Performance (Test Set)")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\n" + "=" * 50)
print("Detailed Classification Report")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# Stratified K-Fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    LinearSVC(C=1.0, class_weight='balanced', max_iter=10000, random_state=42),
    review_vectors, labels, cv=cv, scoring='f1_weighted'
)

print(f"5-Fold Cross-Validation F1 Scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# Confusion matrix visualization
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('LinearSVC Confusion Matrix')
plt.tight_layout()
plt.savefig("data/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to data/confusion_matrix.png")

## 5. Model Comparison: LinearSVC vs Lexicon-Based

In [ ]:
# Apply LinearSVC predictions to ALL reviews
all_predictions = svc_model.predict(review_vectors)
all_pred_labels = le.inverse_transform(all_predictions)

# Get confidence scores from calibrated model
all_confidence = calibrated_svc.predict_proba(review_vectors)
confidence_scores = np.max(all_confidence, axis=1)

# LinearSVC sentiment distribution
svc_dist = pd.Series(all_pred_labels).value_counts()
lexicon_dist = sentiment_df['lexicon_sentiment'].value_counts()

print("Sentiment Distribution Comparison")
print("=" * 50)
comparison = pd.DataFrame({
    'Lexicon-Based': lexicon_dist,
    'LinearSVC': svc_dist
}).fillna(0).astype(int)
print(comparison)
print()

# Agreement between methods
agreement = (all_pred_labels == sentiment_df['lexicon_sentiment'].values).mean()
print(f"Agreement between methods: {agreement:.2%}")

In [ ]:
# Side-by-side comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Lexicon-based distribution
lexicon_dist.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#95a5a6', '#e74c3c'])
axes[0].set_title('Lexicon-Based Sentiment')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# LinearSVC distribution
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
bar_colors = [colors.get(label, '#95a5a6') for label in svc_dist.index]
svc_dist.plot(kind='bar', ax=axes[1], color=bar_colors)
axes[1].set_title('LinearSVC Sentiment')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Sentiment Distribution: Lexicon vs LinearSVC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("data/sentiment_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved to data/sentiment_comparison.png")

In [ ]:
# Show side-by-side predictions for sample reviews
results_sample = pd.DataFrame({
    'Review': sentiment_df['Processed_Review'].str[:80] + '...',
    'Lexicon': sentiment_df['lexicon_sentiment'],
    'LinearSVC': all_pred_labels,
    'Confidence': confidence_scores.round(3),
    'Match': all_pred_labels == sentiment_df['lexicon_sentiment'].values
})

print("Sample Predictions (first 15 reviews):")
print(results_sample.head(15).to_string(index=False))

## 6. Feature Importance Analysis

Analyzing which Word2Vec embedding dimensions contribute most to sentiment classification.

In [ ]:
# LinearSVC coefficients represent importance of each embedding dimension
coef = svc_model.coef_[0]  # Shape: (100,) for binary classification

# Top dimensions contributing to positive and negative sentiment
n_top = 15
top_positive_dims = np.argsort(coef)[-n_top:][::-1]
top_negative_dims = np.argsort(coef)[:n_top]

print("Top Embedding Dimensions for POSITIVE Sentiment:")
print("-" * 45)
for dim in top_positive_dims:
    print(f"  Dimension {dim:3d}: weight = {coef[dim]:+.4f}")

print(f"\nTop Embedding Dimensions for NEGATIVE Sentiment:")
print("-" * 45)
for dim in top_negative_dims:
    print(f"  Dimension {dim:3d}: weight = {coef[dim]:+.4f}")

In [ ]:
# Visualize model weights
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# All dimension weights
axes[0].bar(range(len(coef)), coef, color=['#2ecc71' if c > 0 else '#e74c3c' for c in coef])
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Weight')
axes[0].set_title('LinearSVC Model Weights Across All Embedding Dimensions')
axes[0].axhline(y=0, color='black', linewidth=0.5)

# Top positive and negative dimensions
top_dims = np.concatenate([top_positive_dims, top_negative_dims])
top_weights = coef[top_dims]
top_labels = [f'Dim {d}' for d in top_dims]
bar_colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in top_weights]

axes[1].barh(range(len(top_dims)), top_weights, color=bar_colors)
axes[1].set_yticks(range(len(top_dims)))
axes[1].set_yticklabels(top_labels, fontsize=8)
axes[1].set_xlabel('Weight')
axes[1].set_title(f'Top {n_top} Positive & Negative Sentiment Dimensions')
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig("data/model_weights.png", dpi=150, bbox_inches='tight')
plt.show()
print("Model weights visualization saved to data/model_weights.png")

In [ ]:
# Map embedding dimensions to closest words for interpretability
print("Words Most Associated with Top Sentiment Dimensions")
print("=" * 60)

for sentiment_type, dims in [("POSITIVE", top_positive_dims[:5]), ("NEGATIVE", top_negative_dims[:5])]:
    print(f"\n{sentiment_type} sentiment dimensions:")
    print("-" * 40)
    for dim in dims:
        # Find words whose embeddings have highest values in this dimension
        word_scores = {word: w2v_model.wv[word][dim] for word in w2v_model.wv.index_to_key}
        sorted_words = sorted(word_scores.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
        words_str = ', '.join([f"{w} ({s:.3f})" for w, s in sorted_words])
        print(f"  Dim {dim:3d} (weight={coef[dim]:+.4f}): {words_str}")

## 7. Save Results

In [ ]:
# Save comprehensive results CSV
results_df = pd.DataFrame({
    'Processed_Review': sentiment_df['Processed_Review'],
    'lexicon_score': sentiment_df['lexicon_score'],
    'lexicon_sentiment': sentiment_df['lexicon_sentiment'],
    'linearsvc_prediction': all_pred_labels,
    'linearsvc_confidence': confidence_scores.round(4)
})
results_df.to_csv("data/sentiment_analysis_results.csv", index=False)
print(f"Results saved to data/sentiment_analysis_results.csv ({len(results_df)} reviews)")
print(f"Columns: {list(results_df.columns)}")

In [ ]:
# Save performance metrics to text file
with open("data/model_performance_metrics.txt", "w") as f:
    f.write("LinearSVC + Word2Vec Sentiment Analysis - Performance Metrics\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("Model Configuration\n")
    f.write("-" * 40 + "\n")
    f.write(f"Classifier: LinearSVC (C=1.0, class_weight='balanced')\n")
    f.write(f"Features: Word2Vec embeddings (100 dimensions)\n")
    f.write(f"Word2Vec: CBOW, window=5, min_count=2, epochs=50\n")
    f.write(f"Dataset: {len(sentiment_df)} reviews\n")
    f.write(f"Train/Test split: 80/20 (stratified)\n\n")
    
    f.write("Test Set Performance\n")
    f.write("-" * 40 + "\n")
    f.write(f"Accuracy:  {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall:    {recall:.4f}\n")
    f.write(f"F1-Score:  {f1:.4f}\n\n")
    
    f.write("Cross-Validation (5-Fold Stratified)\n")
    f.write("-" * 40 + "\n")
    f.write(f"Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})\n")
    f.write(f"Fold scores: {', '.join([f'{s:.4f}' for s in cv_scores])}\n\n")
    
    f.write("Classification Report\n")
    f.write("-" * 40 + "\n")
    f.write(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))
    f.write("\n\n")
    
    f.write("Model Comparison\n")
    f.write("-" * 40 + "\n")
    f.write(f"Method agreement: {agreement:.2%}\n")
    lexicon_dict = {k: int(v) for k, v in lexicon_dist.items()}
    f.write(f"Lexicon distribution: {lexicon_dict}\n")
    svc_dict = {k: int(v) for k, v in svc_dist.items()}
    f.write(f"LinearSVC distribution: {svc_dict}\n")
print("Metrics saved to data/model_performance_metrics.txt")

## Summary

This notebook implemented a **LinearSVC classifier with Word2Vec embeddings** for sentiment analysis of skincare reviews:

1. **Word2Vec embeddings** (100-dimensional) were trained on the skincare review corpus using CBOW
2. **Review vectors** were computed by averaging word embeddings for each review
3. **LinearSVC** with balanced class weights was trained to classify sentiment
4. **Model evaluation** showed performance metrics on a held-out test set and via cross-validation
5. **Feature importance** analysis revealed which embedding dimensions drive sentiment predictions
6. Results were compared with the **lexicon-based approach** from Phase 2

### Output Files
- `data/sentiment_analysis_results.csv` – Reviews with predictions from both methods
- `data/model_performance_metrics.txt` – Evaluation metrics
- `data/model_weights.png` – Model weights visualization
- `data/confusion_matrix.png` – Confusion matrix
- `data/sentiment_comparison.png` – Distribution comparison chart